# Template · a sweep notebook for an off-contract model

Copy this file to `1_run_sweep_<yourmodel>.ipynb` and fill in the three `TODO`
blocks. It is a **skeleton, not a runnable notebook**, the same idea as
`../models/_template.py` but for the case where a `_template.py` is not enough.

**When you need this.** Most models don't. If your model returns one logits
tensor `[batch, n_classes]` and takes standard normalised CIFAR, it goes through
`LOCAL_MODELS` in `0_configure_sweeps` and `1_run_sweep_adapt` runs it for you,
no new notebook and no code. Use this template only when the generic
`evaluate()` cannot score your model:

- it returns **several outputs** (an early-exit net with one head per exit, or a
  model with an auxiliary classifier), so "which tensor is the prediction" is a
  choice the sweep cannot make for you; or
- it needs **its own preprocessing**, and the standard per-channel normalised
  CIFAR loaders would give it meaningless accuracy.

Everything else here is deliberately the same as `1_run_sweep_adapt`: the AdaPT
wrap, the int8 calibration, the multiplier list, and the `runs.csv` schema. Keep
it that way. Because you write the same columns into the same
`results/adapt/runs.csv`, `2_accuracy_comparison` and the rest of the analysis
notebooks pick your model up with no changes at all.

**If your model produces a metric that doesn't fit `runs.csv`** (something
per-configuration rather than per-run, say), don't bend the schema. Write the
standard per-model rows into `runs.csv` as usual, put the extra metric in its own
CSV next to it, and give that one its own analysis notebook.

In [ ]:
import csv, time, uuid
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import CIFAR10
from tqdm import tqdm

from adapt.approx_layers.axx_layers import AdaPT_Conv2d
from pytorch_quantization import nn as quant_nn, calib

torch.set_num_threads(40)
%env OMP_PLACES=cores
%env OMP_PROC_BIND=close
%env OMP_WAIT_POLICY=active

## Config

`MODEL_NAME` is both the name of `../models/<MODEL_NAME>.py` and the value that
lands in the `model` column of `runs.csv`, so pick something you're happy to see
in a results table.

The multipliers come from `0_build_multipliers`' manifest rather than a list
here, so a multiplier you add there is swept here too without editing this file.

In [ ]:
import json as _json

# TODO 1/3 — your model.
# ../models/<MODEL_NAME>.py must define build_model(). Unlike the LOCAL_MODELS
# contract it may return anything you like (a tuple of logits, a model with a
# custom forward, ...) — you are the one scoring it further down.
MODEL_NAME = "my_model"
DATASET    = "cifar10"     # only used to label the rows and pick the loaders

BATCH_SIZE    = 128
CALIB_BATCHES = 2          # calibration batches; matches 1_run_sweep_adapt

# Multipliers from 0_build_multipliers' manifest. mul8s_exact is the generator's
# exact twin of the mul8s_acc baseline, so we drop the duplicate and keep acc as
# the exact-int8 reference every drop is measured against.
_manifest = Path("../luts/multipliers.json")
if _manifest.exists():
    _built = _json.loads(_manifest.read_text())["multipliers"]
    MULTS = [m for m in _built if m != "mul8s_exact"] + ["mul8s_acc"]
else:
    MULTS = [f"mul8s_fpga_ish_{i}" for i in range(1, 6)] + ["mul8s_acc"]

# FRESH = False -> resume: rows already in runs.csv are skipped, so you can
#                  interrupt and re-run safely. It only checks that a row exists,
#                  not that it is current -- change the recipe or the LUTs and you
#                  need FRESH = True to actually recompute.
# FRESH = True  -> drop this model's rows and recompute them.
FRESH = False

RESULTS_DIR = Path("../results/adapt")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_DIR / "runs.csv"
COLS = ["run_id", "timestamp", "dataset", "model", "multiplier",
        "retrained_epochs", "top1", "top5", "n_samples", "runtime_s", "notes"]

MODELS_DIR = Path("../models")
if not (MODELS_DIR / f"{MODEL_NAME}.py").exists():
    raise FileNotFoundError(
        f"{MODELS_DIR/f'{MODEL_NAME}.py'} not found. This is a template: point "
        f"MODEL_NAME at your own ../models/<name>.py (see ../models/README.md).")

# FRESH only wipes THIS model's rows. Other models' results live in the same
# CSV, so never just delete the file here.
if FRESH and CSV_PATH.exists():
    _df = pd.read_csv(CSV_PATH)
    _keep = _df[~_df.model.astype(str).str.startswith(MODEL_NAME)]
    print(f"FRESH=True -> dropping {len(_df) - len(_keep)} rows for {MODEL_NAME}")
    _keep.to_csv(CSV_PATH, index=False)

print(f"{MODEL_NAME} on {DATASET} x {len(MULTS)} multipliers = {len(MULTS)} rows")

## Data loaders

Two loaders: one to calibrate on (a couple of batches of *training* data, which
is what sets the int8 ranges) and one to evaluate on.

If your model brings its own preprocessing, this is where it goes. Import the
loader helper from your model's own module rather than reimplementing it here.
The transform that a model was trained under is part of the model, and a
mismatch here silently costs accuracy instead of raising anything.

In [ ]:
# TODO 2/3 — the loaders.
# The default below is plain normalised CIFAR-10, i.e. what 1_run_sweep_adapt
# already does. If that is all you need, you don't need this notebook at all.
# Replace it with your model's own loaders, e.g.
#     from my_model import make_loaders
#     train_set, test_set = make_loaders()
_MEAN, _STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
_tf = T.Compose([T.ToTensor(), T.Normalize(_MEAN, _STD)])
_root = "/workspace/datasets"      # bind-mounted; CIFAR auto-downloads here

train_set = CIFAR10(_root, train=True,  download=True, transform=_tf)
test_set  = CIFAR10(_root, train=False, download=True, transform=_tf)

N_CLASSES = 10

calib_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)
print(f"{len(train_set)} train / {len(test_set)} test images")

## Model + AdaPT wrap

`wrap_with_adapt` walks the module tree and swaps every `nn.Conv2d` for an
`AdaPT_Conv2d` carrying the same weights, so the approximate kernel runs in every
convolution. This is verbatim from `1_run_sweep_adapt`, so copy it rather than
rewrite it. Note it only touches convolutions: the fully-connected layers stay
exact, which is the AdaPT convention this project reports against.

The model is rebuilt from scratch for every multiplier. Wrapping and calibration
both mutate the module in place, so reusing one instance across multipliers would
carry the previous run's quantiser state into the next one.

In [ ]:
def build_model():
    """Import ../models/<MODEL_NAME>.py and call its build_model()."""
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        MODEL_NAME, (MODELS_DIR / f"{MODEL_NAME}.py").resolve())
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.build_model()

def wrap_with_adapt(model, axx_mult):
    """Recursively replace nn.Conv2d -> AdaPT_Conv2d, copying weights/bias."""
    for name, module in model.named_children():
        if isinstance(module, nn.Conv2d) and not isinstance(module, AdaPT_Conv2d):
            new = AdaPT_Conv2d(
                module.in_channels, module.out_channels, module.kernel_size,
                stride=module.stride, padding=module.padding,
                dilation=module.dilation, groups=module.groups,
                bias=module.bias is not None, padding_mode=module.padding_mode,
                axx_mult=axx_mult,
            )
            new.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new.bias.data.copy_(module.bias.data)
            setattr(model, name, new)
        else:
            wrap_with_adapt(module, axx_mult)
    return model

## Calibration + evaluation

The calibration block is boilerplate and is what actually engages the
approximation: AdaPT's layers only route through the LUT once the tensor
quantisers have int8 ranges, so an uncalibrated model quietly gives you exact
float numbers instead of approximate ones.

`evaluate` is the part this whole template exists for. The generic sweep assumes
`model(img)` is the logits. If that is not true for your model, this is where you
say what is.

In [ ]:
def collect_stats(model, loader, num_batches):
    for _, m in model.named_modules():
        if isinstance(m, quant_nn.TensorQuantizer):
            if m._calibrator is not None:
                m.disable_quant(); m.enable_calib()
            else:
                m.disable()
    with torch.no_grad():
        for i, (img, _) in enumerate(loader):
            model(img.cpu())
            if i >= num_batches:
                break
    for _, m in model.named_modules():
        if isinstance(m, quant_nn.TensorQuantizer):
            if m._calibrator is not None:
                m.enable_quant(); m.disable_calib()
            else:
                m.enable()

def compute_amax(model, **kwargs):
    for _, m in model.named_modules():
        if isinstance(m, quant_nn.TensorQuantizer):
            if m._calibrator is not None:
                if isinstance(m._calibrator, calib.MaxCalibrator):
                    m.load_calib_amax()
                else:
                    m.load_calib_amax(**kwargs)
    model.cpu()

def calibrate(model, loader):
    with torch.no_grad():
        collect_stats(model, loader, num_batches=CALIB_BATCHES)
        compute_amax(model, method="percentile", percentile=99.99)


# TODO 3/3 — scoring.
# Turn whatever your model returns into a [batch, n_classes] logits tensor. If it
# has several heads, decide here which one this row is about (and give each head
# its own row, e.g. model="my_model_head1", so they land side by side in the
# comparison tables). Everything below the `logits = ...` line is generic.
def select_logits(out):
    return out

def evaluate(model, loader, n_classes):
    model.eval()
    correct = top5 = total = 0
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    t0 = time.time()
    with torch.no_grad():
        for img, lbl in tqdm(loader, total=len(loader), leave=False):
            logits = select_logits(model(img))
            _, p5 = logits.topk(5, dim=1)
            p1 = p5[:, 0]
            total += lbl.size(0)
            correct += (p1 == lbl).sum().item()
            top5 += (p5 == lbl.unsqueeze(1)).any(dim=1).sum().item()
            for t, p in zip(lbl.numpy(), p1.numpy()):
                cm[t, p] += 1
    dt = time.time() - t0
    return 100 * correct / total, 100 * top5 / total, total, dt, cm

## Writing results

Same schema and the same file as every other AdaPT sweep, which is what buys you
the analysis notebooks for free. `retrained_epochs` stays 0: this template is
eval-only, since the short finetune recipe in `1_run_sweep_adapt` is there as a
control rather than a recommendation.

In [ ]:
def existing_keys():
    if not CSV_PATH.exists():
        return set()
    df = pd.read_csv(CSV_PATH)
    return {tuple(r) for r in df[["dataset", "model", "multiplier", "retrained_epochs"]]
                                  .itertuples(index=False, name=None)}

def append_row(row):
    new_file = not CSV_PATH.exists()
    with CSV_PATH.open("a", newline="") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(COLS)
        w.writerow([row.get(c, "") for c in COLS])

def record(model_name, mult, top1, top5, n, dt, cm):
    short = mult.replace("mul8s_", "")
    d = RESULTS_DIR / "confusion_matrices" / DATASET
    d.mkdir(parents=True, exist_ok=True)
    np.save(d / f"{model_name}_{short}_e0.npy", cm)
    append_row({
        "run_id": uuid.uuid4().hex[:12],
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "dataset": DATASET, "model": model_name, "multiplier": mult,
        "retrained_epochs": 0,
        "top1": f"{top1:.4f}", "top5": f"{top5:.4f}",
        "n_samples": n, "runtime_s": f"{dt:.1f}", "notes": "",
    })

## Sweep

One pass per multiplier: build, wrap, calibrate, evaluate, write the row. Already
present rows are skipped, so an interrupted run picks up where it stopped.

In [ ]:
done = existing_keys()

for mult in MULTS:
    key = (DATASET, MODEL_NAME, mult, 0)
    if key in done:
        print(f"skip {mult} (already in runs.csv)")
        continue

    print(f"\n=== {MODEL_NAME} / {mult} ===")
    model = build_model()
    model = wrap_with_adapt(model, axx_mult=mult)
    calibrate(model, calib_loader)

    top1, top5, n, dt, cm = evaluate(model, test_loader, N_CLASSES)
    record(MODEL_NAME, mult, top1, top5, n, dt, cm)
    print(f"top1={top1:.2f}%  top5={top5:.2f}%  ({n} images, {dt:.1f}s)")

    del model

print("\ndone ->", CSV_PATH.resolve())